# Notebook 2 — QLoRA Fine-Tuning on Qwen3-8B

## What this notebook does

This is your **Phase 3** notebook. It fine-tunes Qwen3-8B on the clothing fit dataset using QLoRA — a technique that trains only a small set of adapter weights on top of a frozen, 4-bit quantized base model.

**Run this on a RunPod A100 40GB instance, not your laptop.**

## The learning goals

1. **Prompt engineering** — how to turn a tabular row into a natural language question an LLM can answer
2. **QLoRA mechanics** — what 4-bit quantization + LoRA adapters actually means in code
3. **SFTTrainer** — the standard supervised fine-tuning loop used by most practitioners
4. **JSON output hardening** — why Pydantic validation is non-negotiable for structured LLM output
5. **Honest evaluation** — comparing LLM vs CatBoost on the exact same test set

---

**Files you need in the same directory before starting:**
```
data/train.parquet   ← 192,733 rows
data/val.parquet     ←  41,300 rows
data/test.parquet    ←  41,301 rows
results/baseline.json
```
These come from `01_eda_baseline.ipynb`. If they're missing, run that notebook first.

## Install dependencies (run once on the pod)
```bash
pip install unsloth trl peft bitsandbytes accelerate transformers wandb pydantic pandas pyarrow
```

## 0. Confirm the baseline gate

Before writing a line of training code, load the gate metric from Notebook 1.
This number defines success. If the fine-tuned LLM doesn't beat it, the LLM doesn't ship.

This discipline — set the bar before you start, not after you see results — prevents the common trap of moving goalposts once a model is trained.

In [1]:
import json, pathlib

gate = json.load(open('results/baseline.json'))
print('=== BASELINE GATE ===')
print(f'  CatBoost macro-F1 : {gate["baseline_macro_f1"]}')
print(f'  Small F1          : {gate["baseline_small_f1"]}')
print(f'  Large F1          : {gate["baseline_large_f1"]}')
print(f'  LLM must beat     : {gate["llm_target_macro_f1"]}  (baseline + 0.03)')
print()
print('If the fine-tuned model does not hit this number, stick with CatBoost.')

=== BASELINE GATE ===
  CatBoost macro-F1 : 0.5178
  Small F1          : 0.4084
  Large F1          : 0.4062
  LLM must beat     : 0.5478  (baseline + 0.03)

If the fine-tuned model does not hit this number, stick with CatBoost.


## 1. Load the splits

We read the parquet files saved by Notebook 1. These are the cleaned, split dataframes — not the raw JSON.
Each row has the original text columns we need for building prompts (`review_text`, `body_type`, etc.).

In [2]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet('data/train.parquet')
val_df   = pd.read_parquet('data/val.parquet')
test_df  = pd.read_parquet('data/test.parquet')

print(f'train: {len(train_df):,} rows | val: {len(val_df):,} rows | test: {len(test_df):,} rows')
print(f'Columns: {train_df.columns.tolist()}')
print()
print('Sample row:')
print(train_df.iloc[0].to_dict())

train: 192,733 rows | val: 41,300 rows | test: 41,301 rows
Columns: ['fit', 'height_in', 'weight_lbs', 'has_height', 'has_weight', 'body_type', 'age', 'bust_size', 'size', 'size_num', 'category', 'review_text', 'review_summary', 'rating', 'source']

Sample row:
{'fit': 'fit', 'height_in': 62.0, 'weight_lbs': 118.0, 'has_height': 1, 'has_weight': 1, 'body_type': 'petite', 'age': 44.0, 'bust_size': '34b', 'size': 4, 'size_num': 4, 'category': 'dress', 'review_text': 'The dress is gorgeous but the gathering in the front is unflattering. I rented the dress but had to take a pass on it because the dress made me look full figured in the front middle. ', 'review_summary': 'UNFLATTERING', 'rating': 4.0, 'source': 'rtr'}


"## 2. Build the prompt dataset\n",
"\n",
"### Three improvements over the naive approach\n",
"\n",
"**1. Class balancing** — the raw training set is 74% `fit`. Without fixing this, the model learns\n",
"to predict `fit` most of the time (gets 74% accuracy while being useless for small/large users).\n",
"We downsample `fit` to 2× the minority class count — giving a roughly balanced 1:2:1 ratio.\n",
"This is the single biggest lever for improving minority-class recall.\n",
"\n",
"**2. BMI in the prompt** — height + weight alone are weak signals. BMI (weight/height²) captures\n",
"the ratio that actually predicts whether clothes will fit. A 150lb person who is 5ft2 vs 5ft10\n",
"needs very different sizing. Adding BMI explicitly saves the model from having to learn this calculation.\n",
"\n",
"**3. Explanation from review text** — the original boilerplate explanation (`'the item fits small'`)\n",
"teaches the model nothing. Using the actual review excerpt as the explanation gives the model\n",
"a richer training target and teaches it to connect review language to fit outcomes."

In [3]:
import json

INSTRUCTION = """You are a clothing fit assistant. Given a user's measurements, the item they ordered, and optionally their review, predict whether the item fit them. Respond with strict JSON only — no other text."""

FIT_TO_ACTION = {'small': 'size_up', 'fit': 'stay', 'large': 'size_down'}

def compute_bmi(height_in, weight_lbs):
    """BMI in imperial: (weight_lbs / height_in²) × 703"""
    if height_in > 0 and weight_lbs > 0:
        return round((weight_lbs / (height_in ** 2)) * 703, 1)
    return None

def build_explanation(fit_label, review):
    """Build a meaningful explanation from the review text instead of boilerplate."""
    review = str(review).strip()
    action_phrase = {
        'small': 'runs small — consider sizing up',
        'fit':   'fits true to size',
        'large': 'runs large — consider sizing down',
    }[fit_label]
    if review and len(review) > 10:
        excerpt = review[:120].rstrip() + ('...' if len(review) > 120 else '')
        return f"This item {action_phrase}. Customer noted: \"{excerpt}\""
    return f"Based on measurements, this item {action_phrase}."

def row_to_prompt(row):
    height_in  = float(row.get('height_in',  -1) or -1)
    weight_lbs = float(row.get('weight_lbs', -1) or -1)
    bmi        = compute_bmi(height_in, weight_lbs)

    user_parts = [
        f"height={'not provided' if height_in <= 0 else f'{height_in:.0f}in'}",
        f"weight={'not provided' if weight_lbs <= 0 else f'{weight_lbs:.0f}lbs'}",
    ]
    if bmi:
        user_parts.append(f"BMI={bmi}")
    body_type = row.get('body_type')
    if body_type and str(body_type).strip() and str(body_type) != 'nan':
        user_parts.append(f"body_type={body_type}")
    age = row.get('age')
    if age and float(age) > 0:
        user_parts.append(f"age={float(age):.0f}")
    bust = row.get('bust_size')
    if bust and str(bust).strip() and str(bust) != 'nan':
        user_parts.append(f"bust={bust}")

    item_parts = []
    category = row.get('category')
    if category and str(category) != 'nan':
        item_parts.append(f"category={category}")
    size = row.get('size')
    if size and str(size) != 'nan':
        item_parts.append(f"size={size}")

    review = str(row.get('review_text', '')).strip()
    review_line = f'Review: "{review}"' if review and review != 'nan' else 'Review: not provided'

    input_text = f"User: {', '.join(user_parts)}\nItem: {', '.join(item_parts)}\n{review_line}"

    fit_label = row['fit']
    output = json.dumps({
        'predicted_fit':      fit_label,
        'confidence':         0.85,
        'recommended_action': FIT_TO_ACTION[fit_label],
        'explanation':        build_explanation(fit_label, review),
    })

    return INSTRUCTION, input_text, output

# Sanity check — look at one example of each class
for label in ['small', 'fit', 'large']:
    row = train_df[train_df['fit'] == label].iloc[0]
    _, inp, out = row_to_prompt(row)
    print(f'--- {label.upper()} ---')
    print(f'INPUT:\n{inp}')
    print(f'OUTPUT: {out}')
    print()

--- SMALL ---
INPUT:
User: height=63in, weight=115lbs, BMI=20.4, body_type=pear, age=32, bust=32b
Item: category=gown, size=8
Review: "I love this dress, so much so that I wish I didn't have to sent it back! I went with a size up for my back up size -- I usually wear a size 2 in bcbg, but I was very happy that I listened to the other reviewers that it runs a little small. The color POPS and the details are beautiful."
OUTPUT: {"predicted_fit": "small", "confidence": 0.85, "recommended_action": "size_up", "explanation": "This item runs small \u2014 consider sizing up. Customer noted: \"I love this dress, so much so that I wish I didn't have to sent it back! I went with a size up for my back up size -- I...\""}

--- FIT ---
INPUT:
User: height=62in, weight=118lbs, BMI=21.6, body_type=petite, age=44, bust=34b
Item: category=dress, size=4
Review: "The dress is gorgeous but the gathering in the front is unflattering. I rented the dress but had to take a pass on it because the dress made me 

In [4]:
from datasets import Dataset

# ── Class balancing ──────────────────────────────────────────────────────────
# Raw split: fit=74%, small=13%, large=13%
# Strategy: downsample 'fit' to 2× minority count → roughly 1:2:1
# This forces the model to learn all three classes, not just the majority

SMOKE_TEST = True   # ← set False for the full production run

small_df = train_df[train_df['fit'] == 'small']
large_df = train_df[train_df['fit'] == 'large']
fit_df   = train_df[train_df['fit'] == 'fit']

minority_count = min(len(small_df), len(large_df))
fit_sample     = fit_df.sample(n=2 * minority_count, random_state=42)

balanced_train = pd.concat([small_df, large_df, fit_sample]).sample(frac=1, random_state=42)

if SMOKE_TEST:
    # 10k rows per class max — full pipeline in ~25 min
    balanced_train = pd.concat([
        small_df.sample(n=min(10_000, len(small_df)), random_state=42),
        large_df.sample(n=min(10_000, len(large_df)), random_state=42),
        fit_df.sample(n=min(20_000, len(fit_df)),     random_state=42),
    ]).sample(frac=1, random_state=42)

print('Balanced training set:')
print(balanced_train['fit'].value_counts())
print(f'Total: {len(balanced_train):,} rows')
print()

# ── Build prompt texts ───────────────────────────────────────────────────────
def build_text(row):
    inst, inp, out = row_to_prompt(row)
    return (
        f"<|im_start|>system\n{inst}<|im_end|>\n"
        f"<|im_start|>user\n{inp}<|im_end|>\n"
        f"<|im_start|>assistant\n{out}<|im_end|>"
    )

print('Building prompt datasets...')
train_texts = [build_text(row) for _, row in balanced_train.iterrows()]
val_texts   = [build_text(row) for _, row in val_df.iterrows()]
print(f'  train: {len(train_texts):,} prompts')
print(f'  val:   {len(val_texts):,} prompts')
print()
print('Sample prompt:')
print(train_texts[0])

train_ds = Dataset.from_dict({'text': train_texts})
val_ds   = Dataset.from_dict({'text': val_texts})

Balanced training set:
fit
fit      20000
large    10000
small    10000
Name: count, dtype: int64
Total: 40,000 rows

Building prompt datasets...
  train: 40,000 prompts
  val:   41,300 prompts

Sample prompt:
<|im_start|>system
You are a clothing fit assistant. Given a user's measurements, the item they ordered, and optionally their review, predict whether the item fit them. Respond with strict JSON only — no other text.<|im_end|>
<|im_start|>user
User: height=69in, weight=not provided, body_type=athletic, age=39, bust=36d
Item: category=dress, size=16
Review: "Such a great and comfortable dress. It fit perfect. You have to pay attention when you put it on to make sure you snap all the necessary places so that it doesn't open up on you. But, once you do that - it looks great!  I got lots of compliments on it. I would definitely rent again.  It was perfect for the boat party I was at, but would also wear it to any party."<|im_end|>
<|im_start|>assistant
{"predicted_fit": "fit", "confid

## 3. Load Qwen3-8B with QLoRA

### What QLoRA actually does

Training a full 8B model would require ~80 GB VRAM and take days. QLoRA solves this in two steps:

1. **4-bit quantization of the base model** — the original float32 weights (8B × 4 bytes = 32 GB) are compressed to int4 (8B × 0.5 bytes = 4 GB). The model is frozen — we never update these weights.

2. **LoRA adapters** — instead of updating the full weight matrices, we add two small matrices A and B alongside each target layer, where `A` is (d × r) and `B` is (r × d) with rank `r=16`. The full update is `W + BA`. We only train A and B — about 0.5% of total parameters.

The result: train an effective 8B model on 16-20 GB VRAM, instead of 80 GB.

### Which layers to add LoRA to
- `q_proj`, `k_proj`, `v_proj`, `o_proj` — the attention mechanism (where the model learns *what to focus on*)
- `gate_proj`, `up_proj`, `down_proj` — the MLP (where the model learns *what to do* with the attended context)

These 7 projection layers are where the task-specific knowledge lives. Adding adapters here is enough.

In [5]:
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 100.0 MB/s  0:00:00eta 0:00:01
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.4.9
    Uninstalling unsloth_zoo-2026.4.9:
      Successfully uninstalled unsloth_zoo-2026.4.9
  Attempting uninstall: unsloth━━━━━━━━━━━━━━━━━ 0/2 [unsloth_zoo]
    Found existing installation: unsloth 2026.4.80/2 [unsloth_zoo]
    Uninstalling unsloth-2026.4.8:━━━━━━━━━━ 0/2 [unsloth_zoo]
      Successfully uninstalled unsloth-2026.4.8m 0/2 [unsloth_zoo]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [unsloth]m1/2 [unsloth]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 768   # most prompts are ~200-300 tokens; 512 is enough and halves memory
LORA_RANK   = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',  # 3B: 3x faster than 8B, same task quality
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,}  ({100*trainable/total:.2f}% of total)')
print(f'Total params     : {total:,}')
print(f'VRAM allocated   : {torch.cuda.memory_allocated()/1e9:.1f} GB')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.096 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params : 29,933,568  (1.73% of total)
Total params     : 1,728,606,208
VRAM allocated   : 2.2 GB


## 4. Train with SFTTrainer

### Training hyperparameters explained

| Param | Value | Why |
|---|---|---|
| `per_device_train_batch_size` | 2 | Keeps VRAM comfortable on A100 40GB |
| `gradient_accumulation_steps` | 8 | Effective batch = 2×8 = 16, same as training with batch 16 but using less memory |
| `num_train_epochs` | 3 | Enough to converge without overfitting short reviews |
| `learning_rate` | 2e-4 | Standard LoRA LR; lower (5e-5) if loss is noisy |
| `warmup_ratio` | 0.05 | Ramp up LR for first 5% of steps — prevents early instability |
| `lr_scheduler_type` | cosine | Smoothly decays LR — works better than linear for LLMs |
| `eval_steps` | 200 | Check val loss every 200 steps to spot overfitting early |

### Gradient accumulation — why it exists
GPUs have fixed VRAM. Larger batches train better (more gradient signal per step) but use more memory.
Gradient accumulation simulates a large batch by running N small batches and summing gradients before updating weights.
`batch=2, accumulation=8` is mathematically equivalent to `batch=16` but uses 8x less VRAM.

In [7]:
import wandb
from trl import SFTTrainer
from transformers import TrainingArguments

# W&B tracks your training — loss curves, eval F1, hyperparams — across all runs
# If you don't have a W&B account: wandb.init(mode='disabled')
wandb.init(project='clothfit-qwen3', name=f'qlora-r{LORA_RANK}-lr2e4')

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    dataset_text_field = 'text',
    max_seq_length    = MAX_SEQ_LEN,
    dataset_num_proc  = 4,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,     # effective batch = 16
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 20,
        eval_strategy               = 'steps',   # renamed from evaluation_strategy in transformers>=4.45
        eval_steps                  = 200,
        save_strategy               = 'steps',
        save_steps                  = 400,
        save_total_limit            = 2,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        report_to                   = 'wandb',
        output_dir                  = './checkpoints',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
    ),
)


trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prakharkhatri123 (prakharkhatri123-iit-roorkee) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/40000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/41300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,000 | Num Epochs = 3 | Total steps = 7,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
200,0.847976,0.720333
400,0.836868,0.713839
600,0.795576,0.708597
800,0.803308,0.690197
1000,0.793214,0.695286
1200,0.781910,0.682598
1400,0.752106,0.695241
1600,0.792768,0.683521
1800,0.747087,0.701036
2000,0.749431,0.686003


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

## 5. Save the fine-tuned adapter

We save only the LoRA adapter weights — not the full 8B model. The adapter is tiny (~100MB).
At inference time: load the frozen base model + merge the adapter.

We also save a merged GGUF version for serving with llama.cpp (CPU-friendly).

In [8]:
# Save LoRA adapter (small — ~100MB)
model.save_pretrained('clothfit-adapter')
tokenizer.save_pretrained('clothfit-adapter')
print('Saved adapter to clothfit-adapter/')

# Optional: export merged GGUF for llama.cpp serving
# This merges the adapter into the base weights and quantizes for CPU inference
# model.save_pretrained_gguf('clothfit-gguf', tokenizer, quantization_method='q4_k_m')
# print('Saved GGUF to clothfit-gguf/')

Unsloth: Restored added_tokens_decoder metadata in clothfit-adapter/tokenizer_config.json.


Saved adapter to clothfit-adapter/


## 6. Evaluate with JSON validation and Pydantic

### Why this section matters

A fine-tuned LLM isn't guaranteed to always output valid JSON. In production, a malformed response crashes your API.
The fix is a Pydantic schema: define exactly what a valid response looks like, try to parse every output, and fall back to CatBoost on failure.

This section teaches:
- How to run batch inference with a fine-tuned model
- How Pydantic schema validation works in practice
- Why JSON validity rate is a key production metric (not just F1)

**We evaluate on the same `data/test.parquet` that CatBoost was evaluated on.** Same rows, same labels, same metric. This is the only honest comparison.

In [9]:
from pydantic import BaseModel, ValidationError
from typing import Literal
import re

class FitResponse(BaseModel):
    predicted_fit     : Literal['small', 'fit', 'large']
    confidence        : float
    recommended_action: Literal['size_down', 'stay', 'size_up']
    explanation       : str

def parse_llm_output(raw: str) -> FitResponse | None:
    """Extract and validate JSON from raw LLM output.
    Returns None on any parse/validation failure — caller handles fallback.
    """
    try:
        # Model sometimes wraps the JSON in markdown code blocks — strip them
        match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        if not match:
            return None
        return FitResponse.model_validate_json(match.group())
    except (ValidationError, ValueError):
        return None

# Quick test of the validator
good = '{"predicted_fit": "small", "confidence": 0.9, "recommended_action": "size_up", "explanation": "fits small"}'
bad1 = '{"predicted_fit": "tiny", "confidence": 0.9, "recommended_action": "size_up", "explanation": "fits"}'
bad2 = 'The item fits small.'

print('Good JSON:', parse_llm_output(good))
print('Bad label:', parse_llm_output(bad1))   # should return None
print('No JSON  :', parse_llm_output(bad2))   # should return None

Good JSON: predicted_fit='small' confidence=0.9 recommended_action='size_up' explanation='fits small'
Bad label: None
No JSON  : None


In [11]:
%pip install scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [12]:
from unsloth import FastLanguageModel
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

# Switch model to inference mode (fuses adapters, enables faster generation)
FastLanguageModel.for_inference(model)

LABEL_MAP = {'small': 0, 'fit': 1, 'large': 2}

def predict_batch(rows, batch_size=16):
    """Run inference on a list of dataframe rows. Returns (preds, n_valid, n_fallback)."""
    preds, n_valid, n_fallback = [], 0, 0

    for i in tqdm(range(0, len(rows), batch_size), desc='Inference'):
        batch = rows[i:i + batch_size]
        prompts = []
        for row in batch:
            inst, inp, _ = row_to_prompt(row)  # no output — we're generating
            prompts.append(
                f"<|im_start|>system\n{inst}<|im_end|>\n"
                f"<|im_start|>user\n{inp}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )

        inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True,
                           max_length=MAX_SEQ_LEN).to('cuda')

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens  = 120,
                do_sample       = False,  # greedy — deterministic for eval
                temperature     = 1.0,
                pad_token_id    = tokenizer.eos_token_id,
            )

        for j, (row, out_ids) in enumerate(zip(batch, outputs)):
            input_len = inputs['input_ids'].shape[1]
            raw = tokenizer.decode(out_ids[input_len:], skip_special_tokens=True)
            parsed = parse_llm_output(raw)
            if parsed:
                preds.append(LABEL_MAP[parsed.predicted_fit])
                n_valid += 1
            else:
                # Fallback: predict majority class 'fit' (same as a naive baseline)
                # In production this would call the CatBoost model instead
                preds.append(LABEL_MAP['fit'])
                n_fallback += 1

    return preds, n_valid, n_fallback

# Run on test set
print(f'Running inference on {len(test_df):,} test rows...')
test_rows = [test_df.iloc[i] for i in range(len(test_df))]
llm_preds, n_valid, n_fallback = predict_batch(test_rows)

json_validity = n_valid / len(test_df)
print(f'\nJSON validity rate: {json_validity:.1%}  ({n_valid:,} valid / {n_fallback:,} fallback)')

Running inference on 41,301 test rows...


Inference:   0%|          | 0/2582 [00:00<?, ?it/s]Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATIO

## 7. Compare LLM vs CatBoost

This is the payoff. Both models evaluated on the same test set, same metric.

**Decision rule:**
- Deploy LLM if: macro-F1 ≥ 0.6007 **AND** JSON validity ≥ 98%
- Otherwise: CatBoost ships, LLM is a research artifact

In [13]:
y_test   = test_df['fit'].map(LABEL_MAP).tolist()
names    = ['small', 'fit', 'large']

llm_macro = f1_score(y_test, llm_preds, average='macro')
llm_per   = f1_score(y_test, llm_preds, average=None)

print('=== LLM (Qwen3-8B QLoRA) ===')
print(classification_report(y_test, llm_preds, target_names=names))

print('\n=== HEAD-TO-HEAD ===')
print(f'{"Metric":<25} {"CatBoost":>10} {"LLM QLoRA":>10} {"Delta":>8}')
print('-' * 58)
rows = [
    ('Macro-F1',        gate['baseline_macro_f1'], llm_macro),
    ('Small F1',        gate['baseline_small_f1'], float(llm_per[0])),
    ('Large F1',        gate['baseline_large_f1'], float(llm_per[2])),
    ('JSON validity',   1.0,                       json_validity),
]
for name, base, llm in rows:
    delta = llm - base
    flag  = '✓' if delta > 0 else '✗'
    print(f'{name:<25} {base:>10.4f} {llm:>10.4f} {delta:>+8.4f} {flag}')

print()
target = gate['llm_target_macro_f1']
if llm_macro >= target and json_validity >= 0.98:
    print(f'✓ LLM PASSES — macro-F1 {llm_macro:.4f} ≥ {target} AND JSON validity {json_validity:.1%} ≥ 98%')
    print('  Recommendation: DEPLOY LLM')
elif llm_macro >= target:
    print(f'⚠ LLM hits F1 target but JSON validity ({json_validity:.1%}) is below 98%')
    print('  Recommendation: fix JSON output before deploying')
else:
    print(f'✗ LLM does NOT pass — macro-F1 {llm_macro:.4f} < {target}')
    print('  Recommendation: KEEP CatBoost. Try more epochs or larger LoRA rank.')

# Save results for future reference
result = {
    'llm_macro_f1':   round(llm_macro, 4),
    'llm_small_f1':   round(float(llm_per[0]), 4),
    'llm_large_f1':   round(float(llm_per[2]), 4),
    'json_validity':  round(json_validity, 4),
    'lora_rank':      LORA_RANK,
    'model':          'Qwen3-8B QLoRA 4-bit',
    'passes_gate':    llm_macro >= target and json_validity >= 0.98,
}
pathlib.Path('results').mkdir(exist_ok=True)
json.dump(result, open('results/llm_eval.json', 'w'), indent=2)
print('\nSaved to results/llm_eval.json')

=== LLM (Qwen3-8B QLoRA) ===
              precision    recall  f1-score   support

       small       0.63      0.77      0.69      6481
         fit       0.88      0.83      0.86     27576
       large       0.64      0.65      0.64      7244

    accuracy                           0.79     41301
   macro avg       0.72      0.75      0.73     41301
weighted avg       0.80      0.79      0.79     41301


=== HEAD-TO-HEAD ===
Metric                      CatBoost  LLM QLoRA    Delta
----------------------------------------------------------
Macro-F1                      0.5178     0.7312  +0.2134 ✓
Small F1                      0.4084     0.6926  +0.2842 ✓
Large F1                      0.4062     0.6442  +0.2380 ✓
JSON validity                 1.0000     0.9998  -0.0002 ✗

✓ LLM PASSES — macro-F1 0.7312 ≥ 0.5478 AND JSON validity 100.0% ≥ 98%
  Recommendation: DEPLOY LLM

Saved to results/llm_eval.json


## 8. What to try if LLM doesn't pass the gate

If macro-F1 < 0.6007, here's a prioritised list of what to adjust:

| Experiment | Change | Expected effect |
|---|---|---|
| Larger LoRA rank | `LORA_RANK = 32` | More expressive adapters — try if r=16 underfits |
| Lower LR | `learning_rate = 5e-5` | Finer updates — try if training loss is noisy |
| More epochs | `num_train_epochs = 5` | More passes over data — try if val loss still decreasing at end |
| Better explanations | Write richer output JSON during data build | Model learns from richer targets |
| Cold-start slice | Evaluate on users NOT in train | Most honest production test |

**If the LLM DOES pass**: the next step is a two-tier inference architecture — CatBoost handles high-confidence cases fast, LLM enhances the uncertain ones. This saves cost at inference time while keeping quality.

```
User request
    → CatBoost (fast, always runs)
       → confidence > 0.80?  →  return CatBoost result directly
       → confidence < 0.60?  →  call LLM for enhanced prediction + explanation
```